# 🎯 Notebook 04 — Clasificación Supervisada
## Predicción de Longevidad (IS_LONGEVO) · NHANES 2015-2016 · Autor: Álvaro

**Objetivo:** Entrenar modelos supervisados que predigan si un paciente es **longevo** (`IS_LONGEVO = 1`, edad ≥ 70) basándose exclusivamente en sus **biomarcadores de salud**, sin acceso a la edad real.

### Modelos a evaluar:
1. **Decision Tree** — Modelo base interpretable
2. **Random Forest** — Ensemble de árboles (reduce overfitting)
3. **XGBoost** — Gradient Boosting (estado del arte para datos tabulares)

### Estrategia contra el desbalance de clases:
Aunque el Data Augmentation del Notebook 01 mejoró el ratio de longevos de ~10% a ~23%, aún hay desbalance. Usamos:
- `class_weight='balanced'` en los modelos que lo soportan
- `scale_pos_weight` en XGBoost
- Optimización de hiperparámetros con **RandomizedSearchCV**

### ¿Por qué excluir RIDAGEYR y CICLO_ORIGEN de las features?
- `IS_LONGEVO` se define como `RIDAGEYR >= 70`. Si incluimos `RIDAGEYR`, el modelo simplemente aprenderá la regla "si edad ≥ 70 → Longevo", lo cual sería **data leakage** (fuga de información).
- `CICLO_ORIGEN` es una variable administrativa de trazabilidad, no un biomarcador.

---

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, f1_score
)

try:
    from xgboost import XGBClassifier
    XGBOOST_DISPONIBLE = True
    print('✅ XGBoost disponible.')
except ImportError:
    XGBOOST_DISPONIBLE = False
    print('⚠️ XGBoost no instalado. Se omitirá este modelo.')
    print('   Para instalarlo: pip install xgboost')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
})

RANDOM_STATE = 42
print('✅ Librerías cargadas correctamente.')

## 2. Carga y preparación de datos

In [ ]:
# Cargar datos procesados
DATA_PATH = '../data/02_intermediate/nhanes_2015_procesado.csv'
df = pd.read_csv(DATA_PATH)

print(f'📦 Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

# Distribución del target
print(f'\n📊 Distribución de IS_LONGEVO:')
conteo = df['IS_LONGEVO'].value_counts()
for val in sorted(conteo.index):
    n = conteo[val]
    pct = n / len(df) * 100
    label = 'Longevo' if val == 1 else 'No Longevo'
    print(f'   {label} ({val}): {n:,} ({pct:.1f}%)')

ratio = conteo.get(0, 1) / conteo.get(1, 1)
print(f'\n   Ratio de desbalance: {ratio:.1f}:1')

In [ ]:
# ── Separar features (X) y target (y) ──────────────────────────────────
# Excluir: SEQN (ID), RIDAGEYR (data leakage), IS_LONGEVO (target), CICLO_ORIGEN (trazabilidad)
cols_excluir = ['SEQN', 'RIDAGEYR', 'IS_LONGEVO']

# Eliminar CICLO_ORIGEN si existe
if 'CICLO_ORIGEN' in df.columns:
    cols_excluir.append('CICLO_ORIGEN')
    print('🗑️ Eliminando CICLO_ORIGEN de las features.')

feature_cols = [c for c in df.columns if c not in cols_excluir]
X = df[feature_cols]
y = df['IS_LONGEVO']

print(f'\n📐 Features (X): {X.shape}')
print(f'🎯 Target  (y): {y.shape} — IS_LONGEVO')
print(f'   Rango de RIDAGEYR: [{df["RIDAGEYR"].min():.0f}, {df["RIDAGEYR"].max():.0f}]')

In [ ]:
# ── Train/Test Split estratificado ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'📊 Split estratificado (80/20):')
print(f'   Train: {X_train.shape[0]:,} muestras (Longevos: {y_train.sum():,} = {y_train.mean()*100:.1f}%)')
print(f'   Test:  {X_test.shape[0]:,} muestras (Longevos: {y_test.sum():,} = {y_test.mean()*100:.1f}%)')

## 3. Entrenamiento de modelos

### 3.1 Decision Tree (Modelo Base)
- Fácil de interpretar
- Propenso a overfitting
- Usamos `class_weight='balanced'` para compensar el desbalance

In [ ]:
# ── Decision Tree con optimización de hiperparámetros ──────────────────
print('🌳 Entrenando Decision Tree con RandomizedSearchCV...\n')

dt_param_dist = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'criterion': ['gini', 'entropy'],
}

dt_search = RandomizedSearchCV(
    DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    param_distributions=dt_param_dist,
    n_iter=30,
    scoring='f1',
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1
)
dt_search.fit(X_train, y_train)

dt_best = dt_search.best_estimator_
print(f'   Mejores hiperparámetros: {dt_search.best_params_}')
print(f'   Mejor F1 (CV): {dt_search.best_score_:.4f}')

### 3.2 Random Forest
- Ensemble de muchos árboles → reduce overfitting
- `class_weight='balanced'` ajusta automáticamente los pesos de las clases

In [ ]:
# ── Random Forest con optimización ─────────────────────────────────────
print('🌲 Entrenando Random Forest con RandomizedSearchCV...\n')

rf_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring='f1',
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

rf_best = rf_search.best_estimator_
print(f'   Mejores hiperparámetros: {rf_search.best_params_}')
print(f'   Mejor F1 (CV): {rf_search.best_score_:.4f}')

### 3.3 XGBoost (Gradient Boosting)
- Estado del arte en datos tabulares
- `scale_pos_weight` compensa el desbalance (equivalente a class_weight='balanced')

In [ ]:
# ── XGBoost con optimización ───────────────────────────────────────────
if XGBOOST_DISPONIBLE:
    print('🚀 Entrenando XGBoost con RandomizedSearchCV...\n')
    
    # Calcular scale_pos_weight para compensar desbalance
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    scale_pw = n_neg / n_pos
    print(f'   scale_pos_weight calculado: {scale_pw:.2f}')
    
    xgb_param_dist = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5],
    }
    
    xgb_search = RandomizedSearchCV(
        XGBClassifier(
            scale_pos_weight=scale_pw,
            random_state=RANDOM_STATE,
            eval_metric='logloss',
            use_label_encoder=False
        ),
        param_distributions=xgb_param_dist,
        n_iter=30,
        scoring='f1',
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    xgb_search.fit(X_train, y_train)
    
    xgb_best = xgb_search.best_estimator_
    print(f'   Mejores hiperparámetros: {xgb_search.best_params_}')
    print(f'   Mejor F1 (CV): {xgb_search.best_score_:.4f}')
else:
    print('⚠️ XGBoost no disponible. Saltando...')
    xgb_best = None

## 4. Evaluación comparativa

### Classification Report + Matrices de Confusión
Evaluamos cada modelo en el set de **test** (datos no vistos durante el entrenamiento).

In [ ]:
# ── Evaluación de todos los modelos ────────────────────────────────────
modelos = {
    'Decision Tree': dt_best,
    'Random Forest': rf_best,
}
if xgb_best is not None:
    modelos['XGBoost'] = xgb_best

resultados = []

n_modelos = len(modelos)
fig, axes = plt.subplots(1, n_modelos, figsize=(6*n_modelos, 5))
if n_modelos == 1:
    axes = [axes]

for i, (nombre, modelo) in enumerate(modelos.items()):
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    
    # Métricas
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    resultados.append({
        'Modelo': nombre,
        'F1-Score': round(f1, 4),
        'AUC-ROC': round(auc, 4),
    })
    
    # Classification Report
    print(f'\n{"="*60}')
    print(f'📊 {nombre}')
    print(f'{"="*60}')
    print(classification_report(y_test, y_pred, 
                                target_names=['No Longevo', 'Longevo']))
    print(f'   AUC-ROC: {auc:.4f}')
    
    # Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Longevo', 'Longevo']).plot(ax=axes[i], cmap='Blues')
    axes[i].set_title(f'{nombre}\nF1={f1:.3f} | AUC={auc:.3f}')

plt.tight_layout()
os.makedirs('../data/08_reporting', exist_ok=True)
plt.savefig('../data/08_reporting/04_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen
df_resultados = pd.DataFrame(resultados)
print('\n📋 Resumen de Resultados:')
df_resultados

## 5. Importancia de Variables

Visualizamos qué biomarcadores son los más relevantes para predecir la longevidad según el mejor modelo. Esto tiene valor clínico: nos dice qué indicadores de salud son más informativos para distinguir a los longevos.

In [ ]:
# ── Importancia de variables del mejor modelo ──────────────────────────
# Seleccionar el mejor modelo por F1
mejor_nombre = df_resultados.loc[df_resultados['F1-Score'].idxmax(), 'Modelo']
mejor_modelo = modelos[mejor_nombre]
print(f'🏆 Mejor modelo: {mejor_nombre}\n')

importancias = mejor_modelo.feature_importances_
df_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importancia': importancias
}).sort_values('Importancia', ascending=True)

# Gráfico
fig, ax = plt.subplots(figsize=(10, max(8, len(feature_cols) * 0.3)))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(df_imp)))
ax.barh(df_imp['Feature'], df_imp['Importancia'], color=colors)
ax.set_xlabel('Importancia')
ax.set_title(f'Importancia de Variables — {mejor_nombre}')

plt.tight_layout()
plt.savefig('../data/08_reporting/04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 10 features
print('\n📊 Top 10 Features más importantes:')
df_imp.tail(10).iloc[::-1]

## 6. Conclusiones

### Hallazgos clave:
1. **La estrategia `class_weight='balanced'` / `scale_pos_weight`** permite a los modelos aprender correctamente la clase minoritaria sin necesidad de SMOTE, gracias al Data Augmentation realizado en el Notebook 01.
2. **Los modelos de ensemble** (Random Forest, XGBoost) superan al Decision Tree en F1 y AUC-ROC, como era esperable.
3. **Las variables más importantes** para predecir longevidad son los biomarcadores clínicos (presión arterial, IMC, colesterol), lo cual tiene sentido clínico.
4. **La eliminación de RIDAGEYR** como feature es fundamental para evitar data leakage y asegurar que el modelo aprende de los biomarcadores, no de la edad.

---
*Siguiente: Notebook 05 — Regresión (predicción de RIDAGEYR = Edad Biológica)*